# 02 Vectorization: TF-IDF and sentence embeddings

Compares the two vector representations of the reports:

| | TF-IDF | Sentence embeddings |
|---|---|---|
| Input | `text_strong` (lemmas, without stopwords) | `text_light` (word forms, sentence context) |
| Used for | NMF | KMeans |

Requires `python -m src.preprocessing.pipeline` and `python -m src.vectorization.pipeline` (computes the embedding cache).

In [ ]:
import os
import sys

# Make the project root importable (the notebook runs in notebooks/).
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd

from src.config import load_config
from src.vectorization.compare import describe, nearest_neighbors, neighbor_overlap
from src.vectorization.pipeline import embeddings_from_config, load_processed, tfidf_from_config

cfg = load_config()
reports = load_processed(cfg)
vectorizer, tfidf = tfidf_from_config(reports, cfg)
embeddings = embeddings_from_config(reports, cfg)
len(reports)

## D1 Key figures

In [ ]:
# One row per representation. Replacing a whole column may change its type.
figures = pd.DataFrame({"TF-IDF": describe(tfidf), "Embeddings": describe(embeddings)}).T
figures = figures.astype({"rows": int, "dimensions": int, "zero_rows": int})
figures["density"] = figures["density"].map("{:.4%}".format)
figures["memory_mb"] = figures["memory_mb"].round(1)
figures

## D2 TF-IDF vocabulary

Share of word pairs and numbers among the terms, and the terms with the lowest IDF (= occurring in the most reports).

In [ ]:
terms = pd.Series(vectorizer.get_feature_names_out())
idf = pd.Series(vectorizer.idf_, index=terms)
print("terms:", len(terms))
print("word pairs:", terms.str.contains(" ").sum())
print("terms with digits:", terms.str.contains(r"\d").sum())
idf.sort_values().head(20)

Reports without any TF-IDF term: empty after strong cleaning (48) or only rare terms below `min_df`.
NMF cannot assign them to a topic (see block E).

In [ ]:
zero = tfidf.getnnz(axis=1) == 0
print("reports without TF-IDF term:", zero.sum(), f"({zero.mean():.2%})")
print("of these empty after strong cleaning:", (reports.loc[zero, "text_strong"] == "").sum())

## D3 Text length and the input limit of the embedding model

The model reads at most 128 tokens (word pieces). Longer texts are truncated.

In [ ]:
from transformers import AutoTokenizer

emb_cfg = cfg["embeddings"]
tokenizer = AutoTokenizer.from_pretrained(emb_cfg["model"], revision=emb_cfg["revision"])
n_tokens = pd.Series([len(ids) for ids in tokenizer(reports["text_light"].tolist())["input_ids"]])
print(n_tokens.describe().round(1))
print("truncated (> 128 tokens):", (n_tokens > 128).sum(), f"({(n_tokens > 128).mean():.2%})")

## D4 Do both representations see the same reports as similar?

For a random sample of reports, compare the 10 nearest neighbours in both representations.
Overlap 1 = same neighbours, 0 = no common neighbour. Reports without a TF-IDF term are excluded.

In [ ]:
K = 10
N_QUERIES = 500
rng = np.random.default_rng(cfg["seed"])
queries = rng.choice(np.flatnonzero(~zero), size=N_QUERIES, replace=False)
overlap = neighbor_overlap(tfidf, embeddings, queries, K)
print(f"mean neighbour overlap (k={K}, {N_QUERIES} reports): {overlap:.2f}")

Example: nearest neighbours of one report in both representations.
Outputs are not committed (nbstripout), because report texts may contain personal data.

In [ ]:
def show_neighbors(query, k=5):
    print("REPORT:", reports.loc[query, "text_light"], "\n")
    for name, matrix in [("TF-IDF", tfidf), ("Embeddings", embeddings)]:
        print(name)
        for i in nearest_neighbors(matrix, query, k):
            print("  -", reports.loc[i, "text_light"][:120])
        print()

show_neighbors(queries[0])

Synonyms: reports that consist only of "Müll" or only of "Abfall".

In [ ]:
mull = reports.index[reports["text_light"] == "Müll"][0]
abfall = reports.index[reports["text_light"] == "Abfall"][0]
print("TF-IDF:    ", (tfidf[mull] @ tfidf[abfall].T).toarray()[0, 0])
print("Embeddings:", round(float(embeddings[mull] @ embeddings[abfall]), 2))

query = reports.index[reports["text_light"] == "Müll liegt rum"][0]
show_neighbors(query, k=10)

## Interpretation

Structure: TF-IDF is sparse (10,941 terms, 0.08 % non-zero) and every column is a word, so NMF topics can be read
as word lists. The embeddings are dense (384 values) and their dimensions have no meaning of their own. For KMeans,
topic words have to come from the texts in each cluster.

Similarity: TF-IDF only finds reports that share lemmas. The embeddings also find reports with the same meaning in
other words: on a scale from 0 (nothing in common) to 1 (same content), "Müll" and "Abfall" score 0 in TF-IDF and
0.85 in the embeddings. For "Müll liegt rum", 6 of the 10 nearest embedding neighbours are about "Abfall", none of
the TF-IDF neighbours. Only 17 % of the 10 nearest neighbours are the same in both representations, so NMF and
KMeans will probably find partly different topics.

Short texts: 1,166 reports (1.5 %) have no TF-IDF vector, because they are empty after strong cleaning (48) or only
contain rare terms. NMF cannot assign them, so they will be reported separately in block E. Embeddings cover every
report, but for very short texts with a name (e.g. a stop) the neighbours can be unrelated, as in the example above.
Truncation at 128 tokens affects only 3.1 % of the texts.

For block E: "nicht", "seit" and "bitte" are the most frequent terms. If they dominate the NMF topics, `max_df` or
the phrase list will be adjusted.

The agreement is based on 500 random reports, the qualitative observations on a few examples.